# Callbacks & Tracing — Custom Handlers, LangSmith & Cost Tracking

Monitor every step of your LangChain pipeline — track tokens, measure latency, log events, and debug with LangSmith.

---

In [ ]:
!pip install -q langchain langchain-openai langsmith

---

## 1 · The Callback System

**The Problem** — Chains, agents, and tools run as black boxes. You can't see what's happening inside.

**The Solution** — Callbacks fire at key lifecycle events. You attach handlers that react to these events.

> Callbacks are event listeners for your AI pipeline. Like `onClick` in a web app, but for LLM calls, tool executions, and chain runs.

In [ ]:
from langchain_openai import ChatOpenAI

# ── verbose=True — the simplest "callback" ──
# This uses a built-in StdOutCallbackHandler that prints events to console

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
chain = prompt | llm | StrOutputParser()

# Invoke with callbacks shown
result = chain.invoke(
    {"topic": "RAG"},
    config={"callbacks": []}   # empty list — we'll add custom handlers next
)
print(f"Result: {result}")

---

## 2 · Custom Callback Handler — Build Your Own

**The Problem** — Built-in handlers don't cover your specific monitoring needs.

**The Solution** — Subclass `BaseCallbackHandler` and override the lifecycle events you care about.

> You decide what to track: latency, tokens, errors, custom metrics, alerting — anything.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
import time
from typing import Any

class DetailedHandler(BaseCallbackHandler):
    """Custom handler that tracks latency, token usage, and events."""

    def __init__(self):
        self.llm_calls = 0
        self.chain_calls = 0
        self.errors = 0
        self.start_time = None

    # ── Chain events ──
    def on_chain_start(self, serialized: dict, inputs: dict, **kwargs: Any) -> None:
        self.chain_calls += 1
        chain_name = serialized.get("name", serialized.get("id", ["Unknown"])[-1])
        print(f"  [CHAIN] ▶ Starting: {chain_name}")

    def on_chain_end(self, outputs: dict, **kwargs: Any) -> None:
        print(f"  [CHAIN] ✅ Completed")

    # ── LLM events ──
    def on_llm_start(self, serialized: dict, prompts: list, **kwargs: Any) -> None:
        self.llm_calls += 1
        self.start_time = time.time()
        print(f"  [LLM]   ▶ Call #{self.llm_calls} starting...")

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        elapsed = time.time() - self.start_time
        # Extract token usage if available
        usage = {}
        if response.llm_output:
            usage = response.llm_output.get("token_usage", {})
        prompt_tokens = usage.get("prompt_tokens", "N/A")
        completion_tokens = usage.get("completion_tokens", "N/A")
        print(f"  [LLM]   ✅ Completed in {elapsed:.2f}s | tokens: {prompt_tokens} prompt + {completion_tokens} completion")

    # ── Error events ──
    def on_llm_error(self, error: Exception, **kwargs: Any) -> None:
        self.errors += 1
        print(f"  [LLM]   ❌ Error: {error}")

    def on_chain_error(self, error: Exception, **kwargs: Any) -> None:
        self.errors += 1
        print(f"  [CHAIN] ❌ Error: {error}")

    def summary(self):
        print(f"\n{'='*50}")
        print(f"Callback Summary:")
        print(f"  Chain calls: {self.chain_calls}")
        print(f"  LLM calls:   {self.llm_calls}")
        print(f"  Errors:      {self.errors}")
        print(f"{'='*50}")

print("DetailedHandler ready.")

In [ ]:
# ── Use the custom handler ──
handler = DetailedHandler()

result = chain.invoke(
    {"topic": "self-attention in transformers"},
    config={"callbacks": [handler]}     # pass handler in config
)

print(f"\nResult: {result}")
handler.summary()

In [ ]:
# ── Run multiple calls with the same handler — metrics accumulate ──

topics = ["embeddings", "vector databases", "fine-tuning with LoRA"]

for topic in topics:
    print(f"\n--- Topic: {topic} ---")
    chain.invoke({"topic": topic}, config={"callbacks": [handler]})

handler.summary()

### What Just Happened?

- Our custom handler intercepted every chain and LLM event
- It tracked latency per LLM call, token usage, and accumulated metrics
- The same handler instance accumulates across multiple calls — useful for session-level metrics
- In production, you'd send these metrics to Datadog, Prometheus, or your monitoring stack

---

## 3 · Token Usage & Cost Tracking

**The Problem** — LLM API calls cost money. Without tracking, you get surprise bills.

**The Solution** — `get_openai_callback` tracks tokens and costs for all OpenAI calls within its scope.

> Wrap your pipeline in a single context manager to get the total cost of the full interaction.

In [ ]:
from langchain_community.callbacks import get_openai_callback

# ── Track a single call ──
with get_openai_callback() as cb:
    result = llm.invoke("What is retrieval-augmented generation?")

print(f"Prompt tokens:     {cb.prompt_tokens}")
print(f"Completion tokens: {cb.completion_tokens}")
print(f"Total tokens:      {cb.total_tokens}")
print(f"Total cost:        ${cb.total_cost:.6f}")
print(f"Successful calls:  {cb.successful_requests}")

In [ ]:
# ── Track multiple calls — costs accumulate ──
with get_openai_callback() as cb:
    chain.invoke({"topic": "transformers"})
    chain.invoke({"topic": "embeddings"})
    chain.invoke({"topic": "vector databases"})

print(f"3 chain calls:")
print(f"  Total tokens: {cb.total_tokens}")
print(f"  Total cost:   ${cb.total_cost:.6f}")
print(f"  Avg cost/call: ${cb.total_cost / 3:.6f}")

In [ ]:
# ── Track an agent with multiple tool calls ──
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '2 + 3 * 4'"""
    try:
        return str(eval(expression))  # safe for demo — use a proper parser in production
    except Exception as e:
        return f"Error: {e}"

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, [calculate], prompt)
executor = AgentExecutor(agent=agent, tools=[calculate], verbose=False)

with get_openai_callback() as cb:
    result = executor.invoke({"input": "What is (15 + 27) * 3, and then divide that by 2?"})

print(f"Agent answer: {result['output']}\n")
print(f"Agent total tokens: {cb.total_tokens}")
print(f"Agent total cost:   ${cb.total_cost:.6f}")
print(f"LLM calls made:     {cb.successful_requests}")
print(f"\n→ Agents make multiple LLM calls (reasoning + tool calls). Costs add up.")

### What Just Happened?

- `get_openai_callback()` tracked every OpenAI API call within its scope
- Multiple chain calls accumulate — useful for session-level cost tracking
- Agents make multiple LLM calls per interaction (reasoning + tool calls), so costs are higher than single calls
- **In production:** log these numbers to detect cost anomalies early

---

## 4 · LangSmith Tracing — Production Observability

**The Problem** — Custom handlers work, but they don't scale. You need a dashboard, historical traces, and team visibility.

**The Solution** — LangSmith provides hosted tracing. Enable it with environment variables.

> LangSmith is to LangChain what Datadog is to web apps — full observability without building it yourself.

In [ ]:
import os

# ── Enable LangSmith tracing (uncomment and fill in your API key) ──
# Sign up for free at: https://smith.langchain.com

# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "your-langsmith-api-key"
# os.environ["LANGSMITH_PROJECT"] = "langchain-tutorials"

print("LangSmith setup:")
print(f"  LANGSMITH_TRACING: {os.environ.get('LANGSMITH_TRACING', 'not set')}")
print(f"  LANGSMITH_PROJECT: {os.environ.get('LANGSMITH_PROJECT', 'not set')}")
print("\nOnce enabled, ALL LangChain calls are automatically traced.")
print("No code changes needed — just set the environment variables.")

In [ ]:
# ── Selective tracing with context manager ──
# Useful when you only want to trace specific calls

try:
    import langsmith as ls

    # Only trace this specific call (even if LANGSMITH_TRACING is off globally)
    # with ls.tracing_context(enabled=True):
    #     chain.invoke({"topic": "semantic search"})

    print("langsmith SDK available. Selective tracing ready.")
    print("Uncomment the code above after setting your API key.")
except ImportError:
    print("langsmith not installed. Run: pip install langsmith")

### What Just Happened?

- LangSmith tracing is **zero-code** — just set environment variables
- Every chain, LLM call, tool execution, and retriever query is automatically captured
- The dashboard shows: traces, latency, token counts, costs, error rates
- Selective tracing lets you trace specific calls without global tracing

**LangSmith provides:**
- Visual trace viewer (parent-child spans)
- Token and cost aggregation
- Latency analysis
- Error tracking
- Dataset and evaluation tools

---

## 5 · Combining Callbacks — Full Monitoring Stack

You can use multiple callback handlers simultaneously.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
import time

class TimingHandler(BaseCallbackHandler):
    """Tracks latency for every LLM call."""
    def __init__(self):
        self.timings = []

    def on_llm_start(self, *args, **kwargs):
        self.start = time.time()

    def on_llm_end(self, *args, **kwargs):
        self.timings.append(time.time() - self.start)

    def report(self):
        if self.timings:
            print(f"  Calls: {len(self.timings)}")
            print(f"  Avg latency: {sum(self.timings)/len(self.timings):.2f}s")
            print(f"  Min: {min(self.timings):.2f}s | Max: {max(self.timings):.2f}s")


class LoggingHandler(BaseCallbackHandler):
    """Logs all events for audit trails."""
    def __init__(self):
        self.log = []

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.log.append({"event": "llm_start", "time": time.time()})

    def on_llm_end(self, response, **kwargs):
        self.log.append({"event": "llm_end", "time": time.time()})

    def on_chain_start(self, serialized, inputs, **kwargs):
        self.log.append({"event": "chain_start", "time": time.time()})

    def on_chain_end(self, outputs, **kwargs):
        self.log.append({"event": "chain_end", "time": time.time()})


# ── Combine: cost tracking + latency + logging ──
timer = TimingHandler()
logger = LoggingHandler()

with get_openai_callback() as cb:
    for topic in ["attention", "embeddings", "fine-tuning"]:
        chain.invoke(
            {"topic": topic},
            config={"callbacks": [timer, logger]}   # multiple handlers
        )

print("=== Cost Report ===")
print(f"  Total tokens: {cb.total_tokens}")
print(f"  Total cost: ${cb.total_cost:.6f}")

print("\n=== Latency Report ===")
timer.report()

print(f"\n=== Audit Log ===")
print(f"  Total events: {len(logger.log)}")
for entry in logger.log[:6]:  # show first 6
    print(f"    {entry['event']:15s} at {entry['time']:.2f}")

### What Just Happened?

- Three monitoring systems ran simultaneously: cost tracking, latency measurement, and event logging
- `get_openai_callback` captured costs, `TimingHandler` captured latency, `LoggingHandler` captured events
- In production, these would feed into dashboards, alerting systems, and audit logs

---

## 6 · Callbacks with Agents — Monitoring Complex Pipelines

In [ ]:
class AgentMonitor(BaseCallbackHandler):
    """Tracks agent behavior: tool calls, iterations, total latency."""

    def __init__(self):
        self.tool_calls = []
        self.llm_calls = 0
        self.start_time = None

    def on_chain_start(self, *args, **kwargs):
        if self.start_time is None:
            self.start_time = time.time()

    def on_llm_start(self, *args, **kwargs):
        self.llm_calls += 1

    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown")
        self.tool_calls.append({"tool": tool_name, "input": input_str, "time": time.time()})
        print(f"  [TOOL] ▶ {tool_name}({input_str[:50]})")

    def on_tool_end(self, output, **kwargs):
        print(f"  [TOOL] ✅ → {str(output)[:80]}")

    def report(self):
        elapsed = time.time() - self.start_time if self.start_time else 0
        print(f"\n{'='*50}")
        print(f"Agent Monitoring Report:")
        print(f"  Total time:   {elapsed:.2f}s")
        print(f"  LLM calls:    {self.llm_calls}")
        print(f"  Tool calls:   {len(self.tool_calls)}")
        for tc in self.tool_calls:
            print(f"    - {tc['tool']}({tc['input'][:40]})")
        print(f"{'='*50}")


# ── Monitor an agent execution ──
monitor = AgentMonitor()

with get_openai_callback() as cb:
    result = executor.invoke(
        {"input": "Calculate (100 + 50) * 2, then divide by 5"},
        config={"callbacks": [monitor]}
    )

print(f"\nAnswer: {result['output']}")
print(f"Cost: ${cb.total_cost:.6f} ({cb.total_tokens} tokens)")
monitor.report()

---

## Key Takeaways

| Monitoring Need | Solution | Effort |
|----------------|----------|--------|
| Quick debugging | `verbose=True` | Zero |
| Token/cost tracking | `get_openai_callback()` | 2 lines |
| Custom metrics | `BaseCallbackHandler` subclass | Medium |
| Production tracing | LangSmith (env vars) | 3 lines |
| Full monitoring | Combine all above | Mix and match |

**Practical advice:**
- Always wrap production calls in `get_openai_callback()` to track costs
- Enable LangSmith in staging and production from day one
- Build custom handlers for metrics specific to your domain
- Monitor agent tool calls closely — they multiply LLM costs

---

## 🎉 Series Complete!

You've covered the full LangChain stack:

| # | Tutorial | What You Learned |
|---|---------|------------------|
| 01 | LangChain Basics | Prompts, chains, batch, streaming |
| 02 | LCEL Deep Dive | RunnableParallel, Lambda, Branch, Fallbacks |
| 03 | Output Parsers | JSON, Pydantic, Enum, auto-fixing |
| 04 | Document Loaders | PDF, CSV, Web ingestion |
| 05 | Text Splitters | Recursive, Token, Semantic chunking |
| 06 | RAG + FAISS | Embeddings, vector search, RAG chain |
| 07 | RAG + Chroma | Persistence, metadata filtering, CRUD |
| 08 | Memory | Buffer, Window, Summary, Entity |
| 09 | Agents & Tools | ReAct, custom tools, tool routing |
| 10 | Callbacks & Tracing | Custom handlers, LangSmith, cost tracking |

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*